# One customer, end to end

This notebook follows **a single person** through the identity-resolution pipeline,
in the order the pipeline actually works:

| | |
|---|---|
| 1 | the raw records, scattered across source systems and disagreeing with each other |
| 2 | what normalisation repairs, and what it cannot |
| 3 | the vector that gets built from each record |
| 4 | the two ways the pipeline looks for candidates — meaning, and exact keys |
| 5 | how those two lists are fused and ranked into one score and one tier |
| 6 | what the model was asked to decide, and what it decided |
| 7 | the merged golden record, and which source won each field |

Nothing here builds anything. Run `./run.sh` first, then work down this notebook.
Every cell runs top to bottom with no edits: the project and the person are read
from `config.env` or chosen by query, never typed in.

> If a cell returns no rows, the pipeline stage above it has not been run yet.

## Setup

Every query below is a real `%%bigquery` cell: the SQL you read *is* the SQL that
runs, and the result comes back as a pandas DataFrame named on the magic line.
Nothing is assembled from strings and nothing shells out to `bq`.

`config.env` is written by `setup.sh` and is the only place a project name lives.
Nothing below hardcodes one — the cell under this one reads it and hands it to the
magic, so every later cell inherits it.

> **Dataset prefix.** A `%%bigquery` cell body is literal text, so the dataset
> cannot be substituted in the way the project can. These cells are written against
> the repo default, `` `cdp.` ``. If your `CDP_DATASET_PREFIX` in `config.env` is
> something else, find-and-replace `` `cdp. `` across this notebook to match.

In [ ]:
%load_ext bigquery_magics

import bigquery_magics
import pandas as pd
from IPython.display import display
from pathlib import Path

# ---------------------------------------------------------------------------
# If you are NOT running from a checkout that has demo/config.env -- a fresh
# clone, Colab, or just a different working directory -- put your project id
# here and run on. Everything else is discovered.
PROJECT = ""          # e.g. "my-project"
DATASET = ""          # leave blank for the default, "cdp"
# ---------------------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 44)

CFG, CONFIG, _source = {}, None, None


def _find_config():
    """demo/config.env, searched upward from wherever this was opened."""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        for candidate in (base / "config.env", base / "demo" / "config.env"):
            if candidate.is_file():
                return candidate
    return None


def _from_gcloud():
    import subprocess
    try:
        r = subprocess.run(["gcloud", "config", "get-value", "project"],
                           capture_output=True, text=True, timeout=20)
        v = (r.stdout or "").strip()
        return v if v and v != "(unset)" else None
    except Exception:
        return None


def _from_adc():
    try:
        import google.auth
        _, project = google.auth.default()
        return project
    except Exception:
        return None


# 1. explicit override
if PROJECT:
    _source = "set in this cell"
else:
    # 2. config.env
    CONFIG = _find_config()
    if CONFIG:
        for _line in CONFIG.read_text().splitlines():
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _v = _line.split("=", 1)
                CFG[_k.strip()] = _v.split("#", 1)[0].strip().strip('"').strip("'")
        PROJECT = CFG.get("CDP_PROJECT", "")
        _source = str(CONFIG)

    # 3-5. environment, gcloud, ADC
    if not PROJECT:
        import os
        for _env in ("GOOGLE_CLOUD_PROJECT", "GCP_PROJECT"):
            if os.environ.get(_env):
                PROJECT, _source = os.environ[_env], f"${_env}"
                break
    # Last resort. These point wherever your gcloud happens to be aimed,
    # which is very often NOT the project holding this demo, so the guess
    # is verified below before anything is queried.
    if not PROJECT and (_p := _from_gcloud()):
        PROJECT, _source = _p, "gcloud default (a guess)"
    if not PROJECT and (_p := _from_adc()):
        PROJECT, _source = _p, "application default credentials (a guess)"

if not PROJECT:
    raise SystemExit(
        "Could not work out which project to use.\n"
        "Set PROJECT at the top of this cell and run it again — you do NOT "
        "need to run setup.sh to read results."
    )

DATASET = DATASET or CFG.get("CDP_DS") or CFG.get("CDP_DATASET_PREFIX") or "cdp"
LOCATION = CFG.get("CDP_LOCATION", "US")

bigquery_magics.context.project = PROJECT
bigquery_magics.context.location = LOCATION
bigquery_magics.context.progress_bar_type = None      # keep the output clean

# Confirm the resolved project really holds this dataset. Without this a
# wrong guess surfaces three cells later as a baffling pandas AttributeError
# on a QueryJob, instead of saying which project it tried.
try:
    from google.cloud import bigquery as _bq
    _bq.Client(project=PROJECT, location=LOCATION).get_dataset(f"{PROJECT}.{DATASET}")
except Exception as _e:
    raise SystemExit(
        f"Project '{PROJECT}' (resolved from {_source}) has no dataset "
        f"'{DATASET}' in location '{LOCATION}'.\n"
        f"  {type(_e).__name__}\n\n"
        "Set PROJECT at the top of this cell to the project holding the demo "
        "and run again.\nYou do NOT need to run setup.sh to read results."
    )

print(f"project    {PROJECT}   (from {_source})")
print(f"dataset    {DATASET}   (location {LOCATION})")
if not CONFIG:
    print("config     not found — reading results only, which is all this "
          "notebook does")
if DATASET != "cdp":
    print(f"\n!! The SQL below is written against `cdp.` but this resolves to "
          f"`{DATASET}.` — find-and-replace before running on.")


def show_record(row, cols=None, wrap=88):
    # Print one wide row as label/value lines. Formatting only, no querying.
    import textwrap
    cols = [c for c in (cols or list(row.index)) if c in row.index]
    width = max(len(c) for c in cols)
    for c in cols:
        v = row[c]
        text = "" if v is None else str(v)
        lines = textwrap.wrap(text, wrap) or [""]
        print(f"{c.rjust(width)}  {lines[0]}")
        for extra in lines[1:]:
            print(f"{' ' * width}  {extra}")


In [ ]:
%%bigquery df_corpus
SELECT (SELECT COUNT(*)                        FROM `cdp.party_records`)    AS source_records,
       (SELECT COUNT(DISTINCT source_system)   FROM `cdp.party_records`)    AS source_systems,
       (SELECT COUNT(DISTINCT person_id)       FROM `cdp.person_assignment`) AS resolved_people

In [ ]:
_c = df_corpus.iloc[0]
print(f"corpus     {_c['source_records']} records from "
      f"{_c['source_systems']} source systems, resolved into "
      f"{_c['resolved_people']} people")

## 1 · Pick the customer

We want one person whose story is worth telling: several records, spread over at
least three different systems, and at least one of those records coming from an
unstructured source — a call transcript or a support ticket, where the identity had
to be *read out of free text* rather than looked up in a column.

The corpus is regenerated from scratch on every run, so the person is **chosen by
query**, not named here. If the strict criteria find nobody, the selection relaxes
step by step rather than failing.

In [ ]:
%%bigquery df_subject
WITH members AS (
  SELECT person_id, record_id FROM `cdp.person_assignment`
),
stats AS (
  SELECT m.person_id,
         COUNT(*)                                        AS n_records,
         COUNT(DISTINCT pr.source_system)                AS n_sources,
         STRING_AGG(DISTINCT pr.source_system ORDER BY pr.source_system) AS sources,
         COUNTIF(pr.source_system IN ('CALL','SUPPORT')) AS n_unstructured
  FROM members AS m
  JOIN `cdp.party_records` AS pr USING (record_id)
  GROUP BY m.person_id
),
-- Did any pair inside this cluster have to go to the adjudicator?
grey AS (
  SELECT a.person_id, COUNT(*) AS n_grey
  FROM `cdp.pair_tiers` AS t
  JOIN members AS a ON a.record_id = t.record_id_a
  JOIN members AS b ON b.record_id = t.record_id_b AND b.person_id = a.person_id
  WHERE t.tier = 'GREY_ZONE'
  GROUP BY a.person_id
),
-- The three criteria, strictest first. A cluster keeps the lowest tier it
-- qualifies for; the ORDER BY then relaxes step by step rather than failing.
ranked AS (
  SELECT s.*,
         IFNULL(g.n_grey, 0) AS grey_zone_pairs,
         CASE
           WHEN s.n_sources >= 3 AND s.n_unstructured >= 1
                AND s.n_records BETWEEN 4 AND 8 THEN 1
           WHEN s.n_sources >= 3 AND s.n_records BETWEEN 4 AND 8 THEN 2
           WHEN s.n_sources >= 2 AND s.n_records > 1 THEN 3
         END AS selection_tier
  FROM stats AS s
  LEFT JOIN grey AS g USING (person_id)
)
SELECT r.person_id,
       CASE r.selection_tier
         WHEN 1 THEN '3+ systems, 4-8 records, at least one unstructured source'
         WHEN 2 THEN '3+ systems, 4-8 records'
         WHEN 3 THEN '2+ systems, more than one record'
       END AS selected_on,
       gp.full_name, gp.city, gp.postcode,
       r.n_records, r.n_sources, r.sources, r.n_unstructured, r.grey_zone_pairs
FROM ranked AS r
LEFT JOIN `cdp.golden_person` AS gp USING (person_id)
WHERE r.selection_tier IS NOT NULL
-- Deterministic: strictest criterion first, then prefer a cluster the
-- adjudicator actually had to rule on, then breadth of sources, then size.
-- person_id breaks any remaining tie.
ORDER BY r.selection_tier,
         IF(r.grey_zone_pairs > 0, 1, 0) DESC,
         r.n_sources DESC, r.n_unstructured DESC, r.n_records DESC, r.person_id
LIMIT 1

In [ ]:
if df_subject.empty:
    raise SystemExit("No multi-record clusters found — has the pipeline been run?")

SUBJECT   = df_subject.iloc[0]
PERSON_ID = str(SUBJECT["person_id"])
params    = {"pid": PERSON_ID}          # passed to every cell below as --params

print(f"selected on : {SUBJECT['selected_on']}\n")
show_record(SUBJECT, cols=["person_id", "full_name", "city", "postcode",
                           "n_records", "n_sources", "sources",
                           "n_unstructured", "grey_zone_pairs"])
print(f"\nEverything below follows {SUBJECT.get('full_name') or PERSON_ID} "
      f"({PERSON_ID}) and nobody else.")

## 2 · The raw records

Here is the customer as the business actually holds them today: one row per system
that has ever seen them. Read down the `raw_name` column first.

Nothing has been cleaned yet. Names are abbreviated, misspelled or simply different;
postcodes are truncated or mistyped; phone numbers are written in whatever format the
system that captured them happened to use; and most systems hold only a fraction of
the fields. Two records can be the same human and share almost no characters.

The counts printed under the table are computed from the rows above them, so they
describe this customer rather than a generalisation.

`source_trust` is the operator's own ranking of how reliable each system is — it
matters later, when the pipeline has to choose between two values that disagree.

In [ ]:
%%bigquery df_raw --params $params
SELECT pr.record_id, pr.source_system, pr.source_trust, pr.source_natural_key,
       pr.raw_name, pr.raw_address, pr.raw_city, pr.raw_postcode,
       pr.raw_email, pr.raw_phone, pr.raw_dob, pr.account_number
FROM `cdp.person_assignment` AS pa
JOIN `cdp.party_records`     AS pr USING (record_id)
WHERE pa.person_id = @pid
ORDER BY pr.source_trust DESC, pr.source_system, pr.record_id
LIMIT 30

In [ ]:
display(df_raw)

# Counted, not asserted: how much do these records actually disagree?
for _field in ("raw_name", "raw_email", "raw_phone", "raw_postcode",
               "raw_dob", "account_number"):
    _vals = sorted({v for v in df_raw[_field].dropna() if str(v).strip()})
    _missing = len(df_raw) - len(df_raw[_field].dropna())
    print(f"{_field:<16} {len(_vals)} distinct value(s), "
          f"{_missing} of {len(df_raw)} records leave it blank"
          + (f"  ->  {_vals}" if 0 < len(_vals) <= 6 else ""))

## 3 · Normalisation — what cleaning can fix, and what it cannot

Stage 20 puts every record through the same small set of functions: fold accents and
punctuation out of names, force phone numbers to international `+61…` form, strip
the noise out of postcodes, lowercase and de-alias emails.

This is the cheap, deterministic half of the problem, and it genuinely helps —
`0419 322 737` and `+61419322737` become the same string, so a plain equality test
now finds them.

It is also where the limit shows. Normalisation repairs *formatting*. It cannot
repair *content*: a misspelled name stays misspelled, a truncated postcode stays
truncated, and a blank field stays blank. Everything that survives this cell is the
hard part, and is why the rest of the pipeline exists.

`identity_strength` counts how many strong identifiers (email, phone, date of birth,
account number) a record actually carries — 0 means there is almost nothing here to
match on.

In [ ]:
%%bigquery df_norm --params $params
SELECT pr.record_id, pr.source_system,
       pr.raw_name,     pr.name_norm,
       pr.raw_postcode, pr.postcode_norm,
       pr.raw_phone,    pr.phone_e164,
       pr.raw_email,    pr.email_norm,
       pr.dob, pr.account_number, pr.identity_strength
FROM `cdp.person_assignment` AS pa
JOIN `cdp.party_records`     AS pr USING (record_id)
WHERE pa.person_id = @pid
ORDER BY pr.identity_strength DESC, pr.record_id
LIMIT 30

In [ ]:
display(df_norm)

print("what the normalisers changed on this customer")
for _raw, _norm, _label in (("raw_name", "name_norm", "name"),
                            ("raw_phone", "phone_e164", "phone"),
                            ("raw_postcode", "postcode_norm", "postcode"),
                            ("raw_email", "email_norm", "email")):
    _has_raw    = df_norm[_raw].notna()  & (df_norm[_raw].astype(str)  != "")
    _has_norm   = df_norm[_norm].notna() & (df_norm[_norm].astype(str) != "")
    _rewritten  = int((_has_raw & _has_norm & (df_norm[_raw] != df_norm[_norm])).sum())
    _dropped    = int((_has_raw & ~_has_norm).sum())
    _blank      = int((~_has_raw).sum())
    print(f"  {_label:<9} rewritten {_rewritten}   "
          f"rejected as unusable {_dropped}   already blank {_blank}")

print("\nstill different after cleaning:")
print("  names    ", sorted(df_norm["name_norm"].dropna().unique()))
print("  emails   ", sorted(df_norm["email_norm"].dropna().unique()))
print("  accounts ", sorted(df_norm["account_number"].dropna().unique()))

# The strongest record becomes the probe for both searches further down.
PROBE_RECORD = str(df_norm.iloc[0]["record_id"])   # highest identity_strength
params_probe_record = {"rid": PROBE_RECORD}
print(f"\nprobe record  {PROBE_RECORD}")

## 4 · The vector

Each record is condensed into one `match_key` string — name, address, postcode,
email, phone, account number, whatever that record happens to have — and that
string is embedded into a vector by a generated column on `party_search`.

A vector is just a long list of numbers that positions the record in a space where
"close together" means "means something similar". It is what lets the pipeline see
that `DYYLAN KITH` and `DYLAN KEITH` are the same idea, when no exact-match rule
ever could.

We take the customer's strongest record as the probe for the two searches that
follow. Only the first few components are printed — there are thousands.

In [ ]:
%%bigquery df_vector --params $params_probe_record
SELECT ps.record_id, ps.source_system, ps.match_key,
       ARRAY_LENGTH(ps.match_embedding.result)             AS dimensions,
       IFNULL(NULLIF(ps.match_embedding.status, ''), 'ok') AS embed_status,
       (SELECT ROUND(SQRT(SUM(v * v)), 4)
        FROM UNNEST(ps.match_embedding.result) AS v)       AS vector_length,
       (SELECT STRING_AGG(FORMAT('%+.4f', v), '  ' ORDER BY o)
        FROM UNNEST(ps.match_embedding.result) AS v WITH OFFSET AS o
        WHERE o < 8)                                       AS first_8_components
FROM `cdp.party_search` AS ps
WHERE ps.record_id = @rid

In [ ]:
VEC   = df_vector.iloc[0]
PROBE = VEC["match_key"]
params_probe = {"probe": PROBE}        # the two searches below both use this

print("the string that was embedded")
print(f"  {PROBE}\n")
print(f"dimensions      {VEC['dimensions']}")
print(f"embed status    {VEC['embed_status']}")
print(f"vector length   {VEC['vector_length']}   (normalised, so cosine is a pure angle)")
print(f"first 8 of {VEC['dimensions']}  {VEC['first_8_components']}  …")
print(f"\nmodel           {CFG.get('CDP_EMBEDDING_ENDPOINT')}")

## 5 · Two ways to look for the same person

The pipeline searches for candidates twice, because the two methods fail in
completely different ways.

**Semantic search** compares vectors. It copes with misspellings, nicknames and
missing fields, because it is matching meaning rather than characters. It is also
blind to identifiers: two *different* account numbers look almost identical to an
embedding, and it will happily return a stranger who lives on a similar-sounding
street. Expect to see exactly that in the results below.

**Keyword / BM25 search** compares the actual tokens. It is exact and cheap and it
never confuses two account numbers — but it finds nothing at all when the name is
misspelled or the shared field is blank.

Run below: the same probe string, both ways.

In [ ]:
%%bigquery df_semantic --params $params_probe
-- Semantic leg. Same body as the `tf_lookup_vector` table function.
SELECT base.record_id, base.source_system, base.match_key,
       ROUND(distance, 4) AS distance          -- 0.0 = identical
FROM AI.SEARCH(
       TABLE `cdp.party_search`,
       'match_key',
       @probe,
       top_k => 10,
       mode  => 'VECTOR')
ORDER BY distance

In [ ]:
display(df_semantic)
print(f"{len(df_semantic)} candidates returned by the semantic leg, "
      f"ranked by cosine distance (0.0 = identical).")

### The keyword leg

Same index, different mode. `HYBRID` runs BM25 over the stored `match_key` text and
the vector search together, and fuses the two rankings inside BigQuery.

One practical note, visible in the SQL: the probe arrives as a **query parameter**
(`@probe`, supplied by `--params` on the magic line) rather than through the shipped
`tf_lookup_hybrid` table function. AI.SEARCH in hybrid mode requires the query string
to be constant-foldable; a query parameter is, but a table-function parameter is not,
so calling the function fails with *"query_value argument of AI.SEARCH must be a
non-null STRING for hybrid search"*. The body below is otherwise identical to the
shipped function.

In [ ]:
%%bigquery df_hybrid --params $params_probe
-- Keyword + vector in one call, fused by the index (BM25 over match_key).
SELECT base.record_id, base.source_system, base.match_key,
       ROUND(distance, 4) AS distance
FROM AI.SEARCH(
       TABLE `cdp.party_search`,
       'match_key',
       @probe,
       top_k => 10,
       mode  => 'HYBRID')
ORDER BY distance

In [ ]:
display(df_hybrid)

_sem_ids = list(df_semantic["record_id"])
_hyb_ids = list(df_hybrid["record_id"])
print(f"probe record        {PROBE_RECORD}")
print(f"returned by both    {sorted(set(_sem_ids) & set(_hyb_ids))}")
print(f"semantic leg only   {sorted(set(_sem_ids) - set(_hyb_ids))}")
print(f"keyword leg only    {sorted(set(_hyb_ids) - set(_sem_ids))}")

### Which leg found which candidate

The live searches above are one probe. The pipeline does this for every record, and
for each surviving pair it records whether the *deterministic key* leg found it
(shared email, phone, account number, postcode + soundex), whether the *semantic*
leg found it, or both — and at what rank in each list.

That is the column to read below: `retrieved_by`. A pair marked `SEMANTIC_ONLY` is
one that no deterministic rule would ever have proposed — the records share no key
at all. A pair marked `LEXICAL_ONLY` is one the embedding missed.

The counts below cover **every** pair touching this customer, not just the top of
the list, so the split is the real one.

In [ ]:
%%bigquery df_leg_counts --params $params
WITH members AS (
  SELECT record_id, person_id FROM `cdp.person_assignment`
),
mine AS (
  SELECT t.*
  FROM `cdp.pair_tiers` AS t
  JOIN members AS ma ON ma.record_id = t.record_id_a
  JOIN members AS mb ON mb.record_id = t.record_id_b
  WHERE ma.person_id = @pid OR mb.person_id = @pid
)
SELECT retrieved_by,
       COUNT(*)                      AS pairs,
       COUNTIF(tier = 'AUTO_MATCH')  AS became_auto_match,
       COUNTIF(tier = 'GREY_ZONE')   AS became_grey_zone,
       COUNTIF(tier = 'REJECT')      AS became_reject
FROM mine
GROUP BY retrieved_by
ORDER BY pairs DESC

In [ ]:
print("every pair touching this customer, by the leg that found it")
display(df_leg_counts)

In [ ]:
%%bigquery df_leg_examples --params $params
-- Examples from each leg, so the single-leg pairs are named rather than counted.
WITH members AS (
  SELECT record_id, person_id FROM `cdp.person_assignment`
),
mine AS (
  SELECT t.*, (ma.person_id = mb.person_id) AS both_in_our_cluster
  FROM `cdp.pair_tiers` AS t
  JOIN members AS ma ON ma.record_id = t.record_id_a
  JOIN members AS mb ON mb.record_id = t.record_id_b
  WHERE ma.person_id = @pid OR mb.person_id = @pid
)
SELECT record_id_a, record_id_b, retrieved_by, both_in_our_cluster,
       matched_keys                 AS keys_the_lexical_leg_matched_on,
       rank_lexical, rank_semantic,
       ROUND(similarity, 3)         AS cosine_similarity,
       ROUND(rrf_score, 5)          AS rrf_score,
       tier
FROM mine
QUALIFY ROW_NUMBER() OVER (PARTITION BY retrieved_by
                           ORDER BY rrf_score DESC, record_id_a) <= 3
ORDER BY retrieved_by, rrf_score DESC
LIMIT 20

In [ ]:
print("up to three examples from each leg")
display(df_leg_examples)

for _leg, _meaning in (("SEMANTIC_ONLY", "share no deterministic key at all — "
                                         "only the embedding connected them"),
                       ("LEXICAL_ONLY", "share an exact key, but the embedding "
                                        "did not rank them near each other")):
    _hits = df_leg_examples[df_leg_examples["retrieved_by"] == _leg]
    if len(_hits):
        print(f"{_leg}: pairs that {_meaning}")
        for _, r in _hits.iterrows():
            print(f"  {r['record_id_a']} ~ {r['record_id_b']}  "
                  f"cosine {r['cosine_similarity']}  -> {r['tier']}")
    else:
        print(f"{_leg}: none for this customer in this run "
              f"(v_retrieval_legs has the corpus-wide split).")
    print()

## 6 · Ranking — turning two lists into one decision

Now the two rankings have to be combined. The problem: a BM25 score and a cosine
distance have no common unit, so they cannot simply be averaged.

**Reciprocal rank fusion** sidesteps that by throwing the scores away and keeping only
the positions: each leg contributes `1 / (60 + its rank)`. A pair that both legs put
near the top wins; a pair that spikes in one leg and is absent from the other does
not. That is `rrf_score`.

Separately, a **rule score** adds up the hard evidence — same account number, same
email, same phone, same date of birth, name similarity — and subtracts for outright
contradictions. That is `rule_score`.

`combined_score` fuses those two, and the tier is then decided against thresholds
from `config.env`, printed below so they are not a mystery:

- at or above `tau_hi` with nothing contradicting → **AUTO_MATCH**, merged with no model call
- clearly below → **REJECT**
- everything in between → **GREY_ZONE**, and only these cost money

`tier_reason` is the plain-English version of whichever rule fired.

In [ ]:
print(f"tau_hi (auto-match at or above) : {CFG.get('CDP_TAU_HI')}")
print(f"tau_lo (grey zone down to)      : {CFG.get('CDP_TAU_LO')}")
print(f"single-signal floor             : {CFG.get('CDP_TAU_SINGLE_SIGNAL')}")
print(f"low-identity floor              : {CFG.get('CDP_TAU_LOW_IDENTITY')}")

In [ ]:
%%bigquery df_ranked --params $params
WITH members AS (
  SELECT record_id, person_id FROM `cdp.person_assignment`
)
SELECT t.record_id_a, t.record_id_b,
       (ma.person_id = mb.person_id)    AS ended_up_same_person,
       t.retrieved_by,
       t.strong_signals,                            -- identifiers that agree
       ROUND(t.rule_score, 3)           AS rule_score,
       ROUND(t.similarity, 3)           AS semantic_similarity,
       ROUND(t.rrf_score, 5)            AS rrf_score,
       ROUND(t.combined_score, 3)       AS combined_score,
       t.forename_conflict,
       t.tier, t.tier_reason
FROM `cdp.pair_tiers` AS t
JOIN members AS ma ON ma.record_id = t.record_id_a
JOIN members AS mb ON mb.record_id = t.record_id_b
WHERE ma.person_id = @pid OR mb.person_id = @pid
ORDER BY t.combined_score DESC
LIMIT 20

In [ ]:
display(df_ranked)

print("why each pair landed where it did")
for _, r in df_ranked.iterrows():
    print(f"  {r['record_id_a']:<15} ~ {r['record_id_b']:<15} "
          f"{r['combined_score']:>6}  {r['tier']:<11} {r['tier_reason']}")

print("\ntier counts for this customer's pairs:",
      df_ranked["tier"].value_counts().to_dict())

## 7 · Adjudication — the only pairs that cost money

Grey-zone pairs are the ones where the evidence genuinely points both ways, and they
are the only ones sent to a model. The prompt carries structured comparison features,
not raw records — the model is asked to weigh stated evidence, not to go fishing.

Every verdict is stamped with the model and prompt version that produced it, and the
ledger is append-only, so any decision can be reproduced later. Read
`decisive_evidence` and `contradiction`: they are the model's own account of what
settled it.

Note that a verdict is not automatically a merge. `UNCERTAIN`, or a `MATCH` below the
acceptance confidence, is parked for a human rather than acted on.

In [ ]:
%%bigquery df_adjudications --params $params
WITH members AS (
  SELECT record_id, person_id FROM `cdp.person_assignment`
),
mine AS (
  SELECT 'this customer' AS scope,
         j.record_id_a, j.record_id_b, t.tier,
         ROUND(t.combined_score, 3) AS combined_score,
         j.verdict, ROUND(j.confidence, 2) AS confidence,
         j.decisive_evidence, j.contradiction, j.rationale,
         j.injection_detected, j.model, j.prompt_version, j.adjudicated_at,
         (ma.person_id = mb.person_id) AS same_cluster
  FROM `cdp.v_adjudications_current` AS j
  JOIN `cdp.pair_tiers` AS t
    ON t.record_id_a = j.record_id_a AND t.record_id_b = j.record_id_b
  JOIN members AS ma ON ma.record_id = j.record_id_a
  JOIN members AS mb ON mb.record_id = j.record_id_b
  WHERE ma.person_id = @pid OR mb.person_id = @pid
),
-- Only populated when this customer never reached the grey zone.
elsewhere AS (
  SELECT 'elsewhere in the corpus' AS scope,
         j.record_id_a, j.record_id_b, t.tier,
         ROUND(t.combined_score, 3) AS combined_score,
         j.verdict, ROUND(j.confidence, 2) AS confidence,
         j.decisive_evidence, j.contradiction, j.rationale,
         j.injection_detected, j.model, j.prompt_version, j.adjudicated_at,
         FALSE AS same_cluster
  FROM `cdp.v_adjudications_current` AS j
  JOIN `cdp.pair_tiers` AS t
    ON t.record_id_a = j.record_id_a AND t.record_id_b = j.record_id_b
  WHERE j.rationale IS NOT NULL
    AND NOT EXISTS (SELECT 1 FROM mine)
  ORDER BY j.confidence DESC, j.record_id_a
  LIMIT 2
)
SELECT * FROM (SELECT * FROM mine UNION ALL SELECT * FROM elsewhere)
ORDER BY same_cluster DESC, confidence DESC
LIMIT 6

In [ ]:
if df_adjudications.empty:
    print("No adjudications recorded in this run at all — stage 60 has not been run.")
else:
    _scope = df_adjudications.iloc[0]["scope"]
    if _scope == "this customer":
        print(f"{len(df_adjudications)} adjudicated pair(s) involve this customer "
              f"(model {df_adjudications.iloc[0]['model']}, "
              f"prompt {df_adjudications.iloc[0]['prompt_version']}).\n")
    else:
        print(f"No pair touching {PERSON_ID} reached the grey zone in this run, so the\n"
              f"adjudicator was never asked about this customer. Every merge here was\n"
              f"settled by the rules alone. Showing an adjudicated pair from elsewhere\n"
              f"in the corpus instead:\n")

    print(f"accept a MATCH at confidence >= {CFG.get('CDP_ACCEPT_CONFIDENCE')}; "
          f"send to a steward below {CFG.get('CDP_STEWARD_CONFIDENCE')}\n")

    for _, r in df_adjudications.iterrows():
        print("=" * 78)
        show_record(r, cols=["record_id_a", "record_id_b", "tier", "combined_score",
                             "verdict", "confidence", "decisive_evidence",
                             "contradiction", "rationale", "injection_detected",
                             "model", "prompt_version", "adjudicated_at"])

## 8 · The match

The accepted pairs become edges in a graph, the graph is resolved into connected
components, and each component becomes one person. Here is the customer's cluster:
every source record that ended up pointing at the same human.

In [ ]:
%%bigquery df_cluster --params $params
SELECT pa.record_id, pr.source_system, pr.source_trust,
       pr.raw_name, pr.name_norm, pr.email_norm, pr.phone_e164,
       pr.account_number, pr.identity_strength
FROM `cdp.person_assignment` AS pa
JOIN `cdp.party_records`     AS pr USING (record_id)
WHERE pa.person_id = @pid
ORDER BY pr.source_trust DESC, pa.record_id
LIMIT 30

In [ ]:
print(f"{PERSON_ID} — {len(df_cluster)} source records, "
      f"{df_cluster['source_system'].nunique()} systems")
display(df_cluster)

### The surviving record

Those records disagree, so something has to choose. Survivorship picks a winner per
field, weighted by how much each source is trusted and how recent the assertion is.
This is the row an agent, a campaign or a service desk would be handed.

In [ ]:
%%bigquery df_golden --params $params
SELECT *
FROM `cdp.golden_person`
WHERE person_id = @pid
LIMIT 1

In [ ]:
if df_golden.empty:
    print("No golden_person row — stage 80 has not been run.")
else:
    show_record(df_golden.iloc[0],
                cols=["person_id", "full_name", "forename", "surname",
                      "address", "city", "postcode", "email", "phone", "dob",
                      "account_number", "populated_fields", "contested_fields",
                      "best_source_trust", "most_recent_assertion"])

### Which source won each field, and what it beat

`was_contested` is the honest column. Where it is true, the sources genuinely
disagreed and the pipeline made a choice — `provenance` records what it chose over,
and `won_from_source` records who won. That trail is what makes the merged record
defensible rather than merely tidy.

In [ ]:
%%bigquery df_survivorship --params $params
SELECT field, surviving_value, won_from_source, source_trust,
       won_from_record_id, asserting_sources, distinct_values,
       was_contested, provenance
FROM `cdp.field_survivorship`
WHERE person_id = @pid
ORDER BY was_contested DESC, field
LIMIT 30

In [ ]:
display(df_survivorship)

_contested = df_survivorship[df_survivorship["was_contested"] == True]  # noqa: E712

print("contested fields, in full")
if _contested.empty:
    print("  none — every source that had an opinion agreed")
for _, r in _contested.iterrows():
    print(f"\n  {r['field']}  ->  {r['surviving_value']}")
    print(f"    {r['provenance']}")

print(f"\nOne person. {len(df_cluster)} records in, 1 record out, "
      f"{len(_contested)} field(s) where a choice had to be made.")

## Scorecard for this run

The customer above is one story. These are the numbers for the whole corpus, marked
against ground truth the pipeline is never allowed to read.

Read `clusters_containing_multiple_people` before anything else: an over-merge and an
under-merge are not interchangeable, because one of them is a privacy incident.
`pct_of_pairs_using_an_llm` is the cost control — if that is not small, the tiering
above it needs attention before the budget does.

In [ ]:
%%bigquery df_scorecard
SELECT * FROM `cdp.v_scorecard`
LIMIT 1

In [ ]:
if df_scorecard.empty:
    print("No scorecard — stage 95 has not been run.")
else:
    show_record(df_scorecard.iloc[0], cols=[
        "records", "true_people", "predicted_people", "people_count_ratio",
        "clusters_containing_multiple_people", "people_split_across_clusters",
        "pairwise_precision", "pairwise_recall", "pairwise_f1", "tp", "fp", "fn",
        "pairs_evaluated", "pairs_sent_to_llm", "pct_of_pairs_using_an_llm",
        "pairs_queued_for_a_human", "pct_of_decisions_deferred",
        "tau_high", "tau_low", "accept_confidence",
        "adjudicator_model", "prompt_version", "embedding_endpoint",
        "generator_seed", "scored_at"])